# 05. Train 실행 노트북

커널을 **Python (blood)** 로 선택한 뒤, 위에서부터 셀을 하나씩 실행하세요 (`Shift + Enter`).

- ① YOLO 학습 → ② 분류기 학습 → ③ 두 방식 비교
- 각 셀은 `code/` 폴더의 .py 파일을 그대로 실행합니다. 코드 내용은 .py 파일에서 바꾸면 됩니다.
- 학습 중에는 노트북 창을 닫지 마세요 (닫으면 학습이 멈출 수 있음).

## 0. 준비 확인 — GPU가 잡히는지

In [18]:
import torch, os
print('PyTorch', torch.__version__)
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
os.chdir(r'D:\05. Blood\05. Train\code')
print('현재 폴더:', os.getcwd())

PyTorch 2.11.0+cu128
GPU 사용 가능: True
GPU: NVIDIA GeForce RTX 5060 Ti
현재 폴더: D:\05. Blood\05. Train\code


## ① YOLO 학습

- 결과: `runs/yolo/yolo11s/weights/best.pt`
- 끝나면 맨 아래에 테스트 mAP50, mAP50-95가 나옵니다.
- GPU 메모리 부족 에러가 나면 끝에 `--batch 8` 추가
- 윈도우 데이터 로더 에러가 나면 끝에 `--workers 0` 추가

In [5]:
!python 10_train_yolo.py --prep "D:/05. Blood/04. Prep" --out "D:/05. Blood/05. Train/runs"

Ultralytics 8.4.184  Python-3.11.16 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16283MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/05. Blood/04. Prep\1_detection_train\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.3, hsv_v=0.2, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s, nbs=64, nms=N

## ② 셀 기반 분류기 학습 (Cellpose 뒤에 붙는 RBC/WBC 분류기)

- 결과: `runs/cell_classifier/best.pt`, `log.csv`
- 끝나면 테스트 정확도와 WBC F1이 나옵니다.

In [7]:
!python 11_train_cell_classifier.py --prep "D:/05. Blood/04. Prep" --out "D:/05. Blood/05. Train/runs"

학습 조각: RBC 11220, WBC 908 / 검증 3640 / 테스트 1832 / 장치 cuda
{'epoch': 1, 'train_loss': 0.0433, 'val_acc': 0.9995, 'val_WBC_precision': 0.9961, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9961}
{'epoch': 2, 'train_loss': 0.0117, 'val_acc': 0.9992, 'val_WBC_precision': 0.9922, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9942}
{'epoch': 3, 'train_loss': 0.0105, 'val_acc': 0.9995, 'val_WBC_precision': 0.9961, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9961}
{'epoch': 4, 'train_loss': 0.0109, 'val_acc': 0.9995, 'val_WBC_precision': 0.9961, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9961}
{'epoch': 5, 'train_loss': 0.008, 'val_acc': 0.9995, 'val_WBC_precision': 0.9961, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9961}
{'epoch': 6, 'train_loss': 0.015, 'val_acc': 0.9992, 'val_WBC_precision': 0.9922, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9942}
{'epoch': 7, 'train_loss': 0.0129, 'val_acc': 0.9995, 'val_WBC_precision': 0.9961, 'val_WBC_recall': 0.9961, 'val_WBC_F1': 0.9961}
{'epoch': 8, 'train_loss': 0

## ③ 두 방식 비교 (평가 0)

- 결과: `results/eval0_stained_test/summary.csv`, `by_source.csv`, `per_image.csv`
- Cellpose는 처음 실행할 때 모델을 자동으로 내려받습니다.

In [8]:
!python 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --classifier "D:/05. Blood/05. Train/runs/cell_classifier/best.pt" --out "D:/05. Blood/05. Train/results/eval0_stained_test"


[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.016
  WBC_precision: 0.985
  WBC_recall: 1.0
  WBC_F1: 0.9924
  RBC_MAE: 3.857
  RBC_precision: 0.77
  RBC_recall: 0.9917
  RBC_F1: 0.8669
  WBC_rel_error: 0.0159
  RBC_rel_error: 0.299
  ms_per_image: 9.3

[Cellpose+classifier]
  method: Cellpose+classifier
  n_images: 126
  WBC_MAE: 0.056
  WBC_precision: 0.9462
  WBC_recall: 0.9389
  WBC_F1: 0.9425
  RBC_MAE: 2.603
  RBC_precision: 0.9152
  RBC_recall: 0.8252
  RBC_F1: 0.8679
  WBC_rel_error: 0.0556
  RBC_rel_error: 0.193
  ms_per_image: 269.3

����옣: D:/05. Blood/05. Train/results/eval0_stained_test



  0%|          | 0.00/1.15G [00:00<?, ?B/s]
  0%|          | 8.00k/1.15G [00:00<7:35:13, 45.2kB/s]
  0%|          | 56.0k/1.15G [00:00<1:35:58, 214kB/s] 
  0%|          | 168k/1.15G [00:00<38:31, 534kB/s]   
  0%|          | 408k/1.15G [00:00<17:39, 1.16MB/s]
  0%|          | 968k/1.15G [00:00<07:50, 2.62MB/s]
  0%|          | 2.18M/1.15G [00:00<03:36, 5.68MB/s]
  0%|          | 3.48M/1.15G [00:00<02:49, 7.24MB/s]
  0%|          | 4.17M/1.15G [00:01<04:11, 4.88MB/s]
  0%|          | 5.53M/1.15G [00:01<03:08, 6.52MB/s]
  1%|          | 6.89M/1.15G [00:01<03:04, 6.64MB/s]
  1%|          | 7.69M/1.15G [00:01<04:10, 4.89MB/s]
  1%|          | 9.05M/1.15G [00:01<03:23, 6.02MB/s]
  1%|          | 10.0M/1.15G [00:02<03:02, 6.71MB/s]
  1%|          | 11.5M/1.15G [00:02<02:22, 8.57MB/s]
  1%|1         | 12.5M/1.15G [00:02<03:44, 5.45MB/s]
  1%|1         | 14.1M/1.15G [00:02<02:45, 7.35MB/s]
  1%|1         | 15.1M/1.15G [00:02<03:40, 5.52MB/s]
  1%|1         | 16.0M/1.15G [00:03<03:23, 5.98MB/s

### 결과 표 보기

In [9]:
import pandas as pd
d = r'D:\05. Blood\05. Train\results\eval0_stained_test'
display(pd.read_csv(os.path.join(d, 'summary.csv')))
display(pd.read_csv(os.path.join(d, 'by_source.csv')))

,method,n_images,WBC_MAE,WBC_precision,WBC_recall,WBC_F1,RBC_MAE,RBC_precision,RBC_recall,RBC_F1,WBC_rel_error,RBC_rel_error,ms_per_image
0,YOLO,126,0.016,0.9850,1.0000,0.9924,3.857,0.7700,0.9917,0.8669,0.0159,0.299,9.3
1,Cellpose+classifier,126,0.056,0.9462,0.9389,0.9425,2.603,0.9152,0.8252,0.8679,0.0556,0.193,269.3


,method,source,n_images,WBC_MAE,WBC_precision,WBC_recall,WBC_F1,RBC_MAE,RBC_precision,RBC_recall,RBC_F1
0,YOLO,BCCD,18,0.000,1.0000,1.0000,1.0000,7.222,0.7312,0.9840,0.8390
1,YOLO,BCDD,14,0.000,1.0000,1.0000,1.0000,8.286,0.6647,1.0000,0.7986
2,YOLO,PBC,50,0.040,0.9636,1.0000,0.9815,2.240,0.8424,0.9936,0.9118
3,YOLO,Raabin-WBC,44,0.000,1.0000,1.0000,1.0000,2.909,0.7747,0.9913,0.8697
4,Cellpose+classifier,BCCD,18,0.333,0.7500,0.6667,0.7059,2.778,0.8325,0.8856,0.8582
5,Cellpose+classifier,BCDD,14,0.000,1.0000,1.0000,1.0000,7.071,0.9333,0.5478,0.6904
6,Cellpose+classifier,PBC,50,0.000,1.0000,1.0000,1.0000,2.500,0.9676,0.8141,0.8842
7,Cellpose+classifier,Raabin-WBC,44,0.023,0.9362,0.9565,0.9462,1.227,0.9221,0.9301,0.9261


## ③-2 YOLO 신뢰도 기준(conf) 정하기

conf가 낮으면 RBC를 너무 많이 셉니다 (conf 0.25일 때 precision 0.77). conf를 바꿔가며 YOLO만 다시 채점합니다 (학습은 다시 안 함).

- **고르기는 val셋으로**, **최종 점수는 test셋으로** 합니다 (test를 보고 고르면 점수가 부풀려짐).
- 고르는 규칙: val에서 **RBC F1이 가장 높은 conf** (단, WBC F1 ≥ 0.98)
- 결과: val에서 **conf 0.7** 선택 → test 최종: RBC MAE 1.206, RBC F1 0.937, WBC F1 1.000

### (1) val셋으로 conf 고르기

In [16]:
for c in [0.25, 0.4, 0.5, 0.6, 0.7, 0.8]:
    !python 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --images "D:/05. Blood/04. Prep/1_detection_train/images/val" --labels "D:/05. Blood/04. Prep/1_detection_train/labels/val" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --conf {c} --out "D:/05. Blood/05. Train/results/val_yolo_conf{c}"


[YOLO]
  method: YOLO
  n_images: 252
  WBC_MAE: 0.02
  WBC_precision: 0.9846
  WBC_recall: 0.9961
  WBC_F1: 0.9903
  RBC_MAE: 3.317
  RBC_precision: 0.7931
  RBC_recall: 0.989
  RBC_F1: 0.8803
  WBC_rel_error: 0.014
  RBC_rel_error: 0.2575
  ms_per_image: 11.2

����옣: D:/05. Blood/05. Train/results/val_yolo_conf0.25

[YOLO]
  method: YOLO
  n_images: 252
  WBC_MAE: 0.012
  WBC_precision: 0.9922
  WBC_recall: 0.9961
  WBC_F1: 0.9942
  RBC_MAE: 2.361
  RBC_precision: 0.8351
  RBC_recall: 0.9807
  RBC_F1: 0.9021
  WBC_rel_error: 0.006
  RBC_rel_error: 0.1876
  ms_per_image: 9.5

����옣: D:/05. Blood/05. Train/results/val_yolo_conf0.4

[YOLO]
  method: YOLO
  n_images: 252
  WBC_MAE: 0.008
  WBC_precision: 0.9961
  WBC_recall: 0.9961
  WBC_F1: 0.9961
  RBC_MAE: 1.841
  RBC_precision: 0.8627
  RBC_recall: 0.976
  RBC_F1: 0.9159
  WBC_rel_error: 0.002
  RBC_rel_error: 0.1487
  ms_per_image: 9.5

����옣: D:/05. Blood/05. Train/results/val_yolo_conf0.5

[YOLO]
  method: YOLO
  n_images: 252
  

In [17]:
import pandas as pd
rows = []
for c in [0.25, 0.4, 0.5, 0.6, 0.7, 0.8]:
    df = pd.read_csv(rf"D:\05. Blood\05. Train\results\val_yolo_conf{c}\summary.csv")
    df.insert(0, "conf", c)
    rows.append(df)
display(pd.concat(rows)[["conf", "RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1", "WBC_MAE", "WBC_F1"]])

,conf,RBC_MAE,RBC_precision,RBC_recall,RBC_F1,WBC_MAE,WBC_F1
0,0.25,3.317,0.7931,0.9890,0.8803,0.020,0.9903
0,0.40,2.361,0.8351,0.9807,0.9021,0.012,0.9942
0,0.50,1.841,0.8627,0.9760,0.9159,0.008,0.9961
0,0.60,1.440,0.8883,0.9682,0.9266,0.008,0.9961
0,0.70,1.119,0.9186,0.9519,0.9350,0.008,0.9961
0,0.80,1.226,0.9504,0.9169,0.9334,0.028,0.9862


### (2) test셋 점수

보고서에는 **conf 0.7 줄만** 최종 성능으로 씁니다. 나머지 줄은 참고용입니다.

In [14]:
for c in [0.25, 0.4, 0.5, 0.6, 0.7, 0.8]:
    !python 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --conf {c} --out "D:/05. Blood/05. Train/results/eval0_yolo_conf{c}"


[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.016
  WBC_precision: 0.985
  WBC_recall: 1.0
  WBC_F1: 0.9924
  RBC_MAE: 3.857
  RBC_precision: 0.77
  RBC_recall: 0.9917
  RBC_F1: 0.8669
  WBC_rel_error: 0.0159
  RBC_rel_error: 0.299
  ms_per_image: 9.8

����옣: D:/05. Blood/05. Train/results/eval0_yolo_conf0.25

[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.008
  WBC_precision: 0.9924
  WBC_recall: 1.0
  WBC_F1: 0.9962
  RBC_MAE: 2.857
  RBC_precision: 0.8156
  RBC_recall: 0.9876
  RBC_F1: 0.8934
  WBC_rel_error: 0.0079
  RBC_rel_error: 0.2228
  ms_per_image: 9.6

����옣: D:/05. Blood/05. Train/results/eval0_yolo_conf0.4

[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.008
  WBC_precision: 0.9924
  WBC_recall: 1.0
  WBC_F1: 0.9962
  RBC_MAE: 2.262
  RBC_precision: 0.8438
  RBC_recall: 0.9822
  RBC_F1: 0.9077
  WBC_rel_error: 0.0079
  RBC_rel_error: 0.1808
  ms_per_image: 9.7

����옣: D:/05. Blood/05. Train/results/eval0_yolo_conf0.5

[YOLO]
  method: YOLO
  n_images: 126
  WB


[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.0
  WBC_precision: 1.0
  WBC_recall: 1.0
  WBC_F1: 1.0
  RBC_MAE: 1.206
  RBC_precision: 0.9129
  RBC_recall: 0.9627
  RBC_F1: 0.9371
  WBC_rel_error: 0.0
  RBC_rel_error: 0.099
  ms_per_image: 10.5

����옣: D:/05. Blood/05. Train/results/eval0_yolo_conf0.7

[YOLO]
  method: YOLO
  n_images: 126
  WBC_MAE: 0.024
  WBC_precision: 1.0
  WBC_recall: 0.9771
  WBC_F1: 0.9884
  RBC_MAE: 1.032
  RBC_precision: 0.9455
  RBC_recall: 0.9254
  RBC_F1: 0.9353
  WBC_rel_error: 0.0238
  RBC_rel_error: 0.0805
  ms_per_image: 9.7

����옣: D:/05. Blood/05. Train/results/eval0_yolo_conf0.8


In [15]:
import pandas as pd
rows = []
for c in [0.25, 0.4, 0.5, 0.6, 0.7, 0.8]:
    df = pd.read_csv(rf"D:\05. Blood\05. Train\results\eval0_yolo_conf{c}\summary.csv")
    df.insert(0, "conf", c)
    rows.append(df)
display(pd.concat(rows)[["conf", "RBC_MAE", "RBC_precision", "RBC_recall", "RBC_F1", "WBC_MAE", "WBC_F1"]])

,conf,RBC_MAE,RBC_precision,RBC_recall,RBC_F1,WBC_MAE,WBC_F1
0,0.25,3.857,0.7700,0.9917,0.8669,0.016,0.9924
0,0.40,2.857,0.8156,0.9876,0.8934,0.008,0.9962
0,0.50,2.262,0.8438,0.9822,0.9077,0.008,0.9962
0,0.60,1.762,0.8737,0.9751,0.9216,0.000,1.0000
0,0.70,1.206,0.9129,0.9627,0.9371,0.000,1.0000
0,0.80,1.032,0.9455,0.9254,0.9353,0.024,0.9884


## ④ 하한 측정 — 비염색 영상을 가상 염색 없이 그대로 넣기

RBC 정답만 있어서 WBC 지표는 의미 없습니다.

- 예전 결과(`eval2_unstained_no_staining`)는 conf 0.25 기준이었습니다. 확정한 **conf 0.7**로 다시 잽니다.

In [19]:
!python 12_eval_compare.py --prep "D:/05. Blood/04. Prep" --images "D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse/images" --labels "D:/05. Blood/04. Prep/4_eval_unstained/redtell_dse/labels" --yolo "D:/05. Blood/05. Train/runs/yolo/yolo11s/weights/best.pt" --conf 0.7 --classifier "D:/05. Blood/05. Train/runs/cell_classifier/best.pt" --out "D:/05. Blood/05. Train/results/eval2_unstained_no_staining_conf0.7"


[YOLO]
  method: YOLO
  n_images: 93
  WBC_MAE: 0.011
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 14.065
  RBC_precision: 0.9385
  RBC_recall: 0.6333
  RBC_F1: 0.7562
  WBC_rel_error: 
  RBC_rel_error: 0.2811
  ms_per_image: 10.6

[Cellpose+classifier]
  method: Cellpose+classifier
  n_images: 93
  WBC_MAE: 20.312
  WBC_precision: 0.0
  WBC_recall: 0.0
  WBC_F1: 0.0
  RBC_MAE: 18.344
  RBC_precision: 0.7723
  RBC_recall: 0.4528
  RBC_F1: 0.5708
  WBC_rel_error: 
  RBC_rel_error: 0.4295
  ms_per_image: 855.6

����옣: D:/05. Blood/05. Train/results/eval2_unstained_no_staining_conf0.7


C:\Users\KonYang\anaconda3\envs\blood\Lib\site-packages\cellpose\dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


### ④ 결과 표 보기

**RBC 열만** 보면 됩니다 (WBC 숫자는 무시).

In [20]:
import pandas as pd, os
d = r"D:\05. Blood\05. Train\results\eval2_unstained_no_staining_conf0.7"
display(pd.read_csv(os.path.join(d, "summary.csv")))

,method,n_images,WBC_MAE,WBC_precision,WBC_recall,WBC_F1,RBC_MAE,RBC_precision,RBC_recall,RBC_F1,WBC_rel_error,RBC_rel_error,ms_per_image
0,YOLO,93,0.011,0.0,0.0,0.0,14.065,0.9385,0.6333,0.7562,NaN,0.2811,10.6
1,Cellpose+classifier,93,20.312,0.0,0.0,0.0,18.344,0.7723,0.4528,0.5708,NaN,0.4295,855.6
